This notebook ingests and prepares data from the UK Government Ethnicity Facts and Figures resource, which compiles evidence on disparities across a range of domains including health, employment, education, and living conditions. The purpose is to extract and structure ethnicity-related indicators relevant to wider determinants of health, providing context beyond clinical outcomes.

In [3]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import importlib.util
from typing import Any

import pandas as pd

HERE = Path.cwd()
PARQUET_ENGINE = "pyarrow" if importlib.util.find_spec("pyarrow") else "fastparquet"

# Existing folder conventions
ETH_MAP_PATH = HERE / "_config_ethnicity_mapping.csv"
OUT_PARQUET_PATH = HERE / "_processed_ethnicity_facts_gp_satisfaction.parquet"
QA_PATH = HERE / "_qa_ethnicity_facts_ingestion_issues.csv"

EFF_URL = (
    "https://www.ethnicity-facts-figures.service.gov.uk/health/patient-experience/"
    "patient-satisfaction-with-gp-services/latest/downloads/by-ethnicity-gp-services-overall.csv"
)

In [4]:
# Defensive helpers

@dataclass
class IngestionIssue:
    issue_type: str
    detail: str
    rows_affected: int | None = None
    columns_involved: str | None = None


def _lower(s: str) -> str:
    return str(s).strip().lower()


def pick_first_matching_column(df: pd.DataFrame, contains_any: list[str]) -> str | None:
    cols = list(df.columns)
    scored: list[tuple[int, str]] = []
    for c in cols:
        cl = _lower(c)
        score = sum(1 for token in contains_any if token in cl)
        if score > 0:
            scored.append((score, c))
    if not scored:
        return None
    scored.sort(key=lambda x: (-x[0], x[1]))
    return scored[0][1]


def pick_value_column(df: pd.DataFrame) -> str | None:
    # Prefer columns literally named "value" or containing percent/pct where numeric.
    preferred = ["value", "val", "percentage", "percent", "pct", "rate"]
    candidate = pick_first_matching_column(df, preferred)
    if candidate is not None and pd.api.types.is_numeric_dtype(df[candidate]):
        return candidate

    # Else: choose a numeric column with the highest non-null proportion, excluding obvious CI bounds.
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    numeric_cols = [
        c for c in numeric_cols
        if not any(tok in _lower(c) for tok in ["lower", "upper", "ci", "conf", "interval"])
    ]
    if not numeric_cols:
        return None

    non_null_rates = {c: df[c].notna().mean() for c in numeric_cols}
    return max(non_null_rates, key=non_null_rates.get)


def safe_read_csv(url: str) -> pd.DataFrame:
    # Be defensive about encoding; pandas usually handles this, but keep errors explicit.
    try:
        return pd.read_csv(url)
    except UnicodeDecodeError:
        return pd.read_csv(url, encoding="utf-8", encoding_errors="replace")


def write_qa(issues: list[IngestionIssue], path: Path) -> None:
    if not issues:
        # Still write an empty QA file so the pipeline is auditable.
        pd.DataFrame(columns=["issue_type", "detail", "rows_affected", "columns_involved"]).to_csv(
            path, index=False
        )
        return

    qa_df = pd.DataFrame(
        [
            {
                "issue_type": i.issue_type,
                "detail": i.detail,
                "rows_affected": i.rows_affected,
                "columns_involved": i.columns_involved,
            }
            for i in issues
        ]
    )
    qa_df.to_csv(path, index=False)

In [6]:
# Load ethnicity mapping (raw -> standard)

issues: list[IngestionIssue] = []

if not ETH_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Missing ethnicity mapping file: {ETH_MAP_PATH.name}. "
        "Expected it in the working directory."
    )

eth_map = pd.read_csv(ETH_MAP_PATH)

required_cols = {"raw_ethnicity", "standard_ethnicity"}
missing = required_cols - set(eth_map.columns)
if missing:
    raise KeyError(
        f"{ETH_MAP_PATH.name} is missing required columns: {sorted(missing)}. "
        f"Found columns: {sorted(eth_map.columns)}"
    )

# Build mapping dict (string-normalised keys)
ETH_MAP = {
    str(raw).strip(): str(std).strip()
    for raw, std in zip(eth_map["raw_ethnicity"], eth_map["standard_ethnicity"])
    if pd.notna(raw) and pd.notna(std)
}

len(ETH_MAP), list(ETH_MAP.items())[:5]

(8,
 [('Black African', 'Black African'),
  ('African', 'Black African'),
  ('Black Caribbean', 'Black Caribbean'),
  ('Caribbean', 'Black Caribbean'),
  ('White British', 'White British')])

In [8]:
# Ingest EFF CSV

df_raw = safe_read_csv(EFF_URL)

if df_raw.empty:
    raise ValueError("EFF CSV downloaded but contains 0 rows.")

df_raw.shape, df_raw.columns.tolist()[:15], df_raw.head()

((21, 2),
 ['Ethnicity', '%'],
      Ethnicity     %
 0          All  83.1
 1  Bangladeshi  73.6
 2      Chinese  77.8
 3       Indian  77.6
 4    Pakistani  73.2)

In [9]:
# Detect key columns defensively

eth_col = pick_first_matching_column(df_raw, ["ethnic"])
time_col = pick_first_matching_column(df_raw, ["time", "year", "period", "date"])
measure_col = pick_first_matching_column(df_raw, ["measure", "metric", "indicator", "question", "topic"])
geo_col = pick_first_matching_column(df_raw, ["geography", "area", "region", "country"])
value_col = pick_value_column(df_raw)

detected = {
    "ethnicity_column": eth_col,
    "time_column": time_col,
    "measure_column": measure_col,
    "geography_column": geo_col,
    "value_column": value_col,
}
detected

{'ethnicity_column': 'Ethnicity',
 'time_column': None,
 'measure_column': None,
 'geography_column': None,
 'value_column': '%'}

In [10]:
# Validate detected columns

if eth_col is None:
    raise KeyError("Could not detect an ethnicity column (looked for tokens like 'ethnic').")

if value_col is None:
    raise KeyError("Could not detect a numeric value column.")

# Convert value column to numeric defensively (if they string)
df = df_raw.copy()
df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

n_value_na = int(df[value_col].isna().sum())
if n_value_na > 0:
    issues.append(
        IngestionIssue(
            issue_type="value_parse_to_numeric",
            detail=f"Coerced non-numeric values to NaN in '{value_col}'.",
            rows_affected=n_value_na,
            columns_involved=value_col,
        )
    )

df.shape

(21, 2)

In [11]:
# Standardise ethnicity

df["raw_ethnicity"] = df[eth_col].astype(str).str.strip()

df["standard_ethnicity"] = (
    df["raw_ethnicity"]
    .map(ETH_MAP)
    .fillna(df["raw_ethnicity"])
)

# QA: unmapped categories
unmapped = df.loc[df["raw_ethnicity"] == df["standard_ethnicity"], "raw_ethnicity"].unique().tolist()

# "Unmapped" is not necessarily bad (could already be standard), so only flag if mapping file
# doesn't contain that category and it looks like a detailed/variant label.
mapping_keys = set(ETH_MAP.keys())
suspect_unmapped = [x for x in unmapped if x not in mapping_keys]

if suspect_unmapped:
    issues.append(
        IngestionIssue(
            issue_type="unmapped_ethnicity_values",
            detail=f"Found ethnicity labels not present in mapping file (showing up to 20): "
                   f"{suspect_unmapped[:20]}",
            rows_affected=None,
            columns_involved=eth_col,
        )
    )

df["standard_ethnicity"].value_counts(dropna=False).head(20)

standard_ethnicity
All                                1
Bangladeshi                        1
Chinese                            1
Indian                             1
Pakistani                          1
Asian other                        1
Black African                      1
Black Caribbean                    1
Black other                        1
Mixed white and Asian              1
Mixed white and black African      1
Mixed white and black Caribbean    1
Mixed other                        1
White British                      1
Irish                              1
Gypsy or Irish traveller           1
Roma                               1
White other                        1
Arab                               1
Any other ethnic background        1
Name: count, dtype: int64

In [33]:
# Build analysis-ready schema (correctly aligned to df index)
out = pd.DataFrame(index=df.index)

out["source"] = "Ethnicity Facts & Figures"
out["dataset"] = "Patient satisfaction with GP services (overall)"
out["geography_level"] = "England"

# Geography: if present, keep it; else set to England.
if geo_col is not None:
    out["geography_name"] = df[geo_col].astype(str).str.strip()
else:
    out["geography_name"] = "England"

# Time: keep raw if present; else set to "latest"
if time_col is not None:
    out["time_period"] = df[time_col].astype(str).str.strip()
else:
    out["time_period"] = "latest"

# Measure / question / topic
if measure_col is not None:
    out["measure_name"] = df[measure_col].astype(str).str.strip()
else:
    out["measure_name"] = "Overall satisfaction"

out["value"] = df[value_col]
out["value_unit"] = "Percent"  # This dataset is percentage satisfied

out["raw_ethnicity"] = df["raw_ethnicity"]
out["standard_ethnicity"] = df["standard_ethnicity"]

# Keep useful supporting columns if present
support_cols = []
for c in df.columns:
    cl = c.lower()
    if any(tok in cl for tok in ["denominator", "numerator", "lower", "upper", "ci", "confidence"]):
        support_cols.append(c)

for c in support_cols:
    out[c] = df[c]

# Drop duplicate ethnicity column if present (source sometimes includes "Ethnicity")
if "Ethnicity" in out.columns:
    out = out.drop(columns=["Ethnicity"])

FOCUS = {"Black African", "Black Caribbean", "White British", "All"}
out["is_focus_group"] = out["standard_ethnicity"].isin(FOCUS)

out.shape, out.head()

((21, 11),
                       source                                          dataset  \
 0  Ethnicity Facts & Figures  Patient satisfaction with GP services (overall)   
 1  Ethnicity Facts & Figures  Patient satisfaction with GP services (overall)   
 2  Ethnicity Facts & Figures  Patient satisfaction with GP services (overall)   
 3  Ethnicity Facts & Figures  Patient satisfaction with GP services (overall)   
 4  Ethnicity Facts & Figures  Patient satisfaction with GP services (overall)   
 
   geography_level geography_name time_period          measure_name  value  \
 0         England        England      latest  Overall satisfaction   83.1   
 1         England        England      latest  Overall satisfaction   73.6   
 2         England        England      latest  Overall satisfaction   77.8   
 3         England        England      latest  Overall satisfaction   77.6   
 4         England        England      latest  Overall satisfaction   73.2   
 
   value_unit raw_ethnici

In [34]:
# Basic validation checks (no joins yet)

# Duplicates check on core grain
core_keys = ["geography_name", "time_period", "measure_name", "standard_ethnicity"]
dup_mask = out.duplicated(subset=core_keys, keep=False)
n_dups = int(dup_mask.sum())
if n_dups > 0:
    issues.append(
        IngestionIssue(
            issue_type="duplicate_rows_on_core_keys",
            detail=f"Found duplicates on keys {core_keys}.",
            rows_affected=n_dups,
            columns_involved=", ".join(core_keys),
        )
    )

# Missingness check for critical fields
critical_cols = ["time_period", "measure_name", "value", "standard_ethnicity"]
for c in critical_cols:
    n_missing = int(out[c].isna().sum())
    if n_missing > 0:
        issues.append(
            IngestionIssue(
                issue_type="missing_critical_field",
                detail=f"Missing values in critical field '{c}'.",
                rows_affected=n_missing,
                columns_involved=c,
            )
        )

# Show quick summary
summary = {
    "rows": len(out),
    "unique_standard_ethnicities": int(out["standard_ethnicity"].nunique(dropna=True)),
    "value_missing": int(out["value"].isna().sum()),
    "duplicates_on_core_keys": n_dups,
}

missing_focus = sorted(FOCUS - set(out["standard_ethnicity"].unique()))
missing_focus

[]

In [35]:
# Save outputs (parquet + QA)

write_qa(issues, QA_PATH)

out.to_parquet(OUT_PARQUET_PATH, index=False, engine=PARQUET_ENGINE)

OUT_PARQUET_PATH, QA_PATH

(WindowsPath('c:/Users/OlenaManziuk/OneDrive - Third Sector Together North West London/Documents/nwl_health_inequalities/_processed_ethnicity_facts_gp_satisfaction.parquet'),
 WindowsPath('c:/Users/OlenaManziuk/OneDrive - Third Sector Together North West London/Documents/nwl_health_inequalities/_qa_ethnicity_facts_ingestion_issues.csv'))

In [36]:
# Quick peek (sanity check)

out.head(10)

,source,dataset,geography_level,geography_name,time_period,measure_name,value,value_unit,raw_ethnicity,standard_ethnicity,is_focus_group
0,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,83.1,Percent,All,All,True
1,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,73.6,Percent,Bangladeshi,Bangladeshi,False
2,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,77.8,Percent,Chinese,Chinese,False
3,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,77.6,Percent,Indian,Indian,False
4,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,73.2,Percent,Pakistani,Pakistani,False
5,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,79.7,Percent,Asian other,Asian other,False
6,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,86.2,Percent,Black African,Black African,True
7,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,83.7,Percent,Black Caribbean,Black Caribbean,True
8,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,83.1,Percent,Black other,Black other,False
9,Ethnicity Facts & Figures,Patient satisfaction with GP services (overall),England,England,latest,Overall satisfaction,78.7,Percent,Mixed white and Asian,Mixed white and Asian,False


The Ethnicity Facts and Figures data has been cleaned, standardised, and reshaped into a consistent, analysis-ready format. Relevant indicators have been aligned with the project’s ethnicity framework and validated for usability. This dataset strengthens the evidence base by incorporating wider social determinants, supporting more comprehensive analysis of inequalities in later stages.